In [1]:
import os

import pandas as pd
import torch
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader
from torchvision.datasets import VisionDataset
from torchvision.transforms import v2

torch.manual_seed(3407)
if torch.cuda.is_available():
    torch.cuda.manual_seed(3407)

cudnn.deterministic = False
cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

test_dir_candidates = [
    "/kaggle/input/cassava-leaf-disease-classification/test_images/",
    "/kaggle/input/cassava-leaf-disease-classification/test_images/test_images/",
    "/kaggle/input/test_images/",
    "/kaggle/input/test_images/test_images/",
    "/kaggle/data/cassava-leaf-disease-classification/test_images/",
    "/kaggle/data/cassava-leaf-disease-classification/test_images/test_images/",
    "/kaggle/data/test_images/",
    "/kaggle/data/test_images/test_images/",
]
test_dir = None
for p in test_dir_candidates:
    if os.path.isdir(p):
        try:
            has_jpg = any(
                os.path.isfile(os.path.join(p, f))
                and f.lower().endswith((".jpg", ".jpeg", ".png"))
                for f in os.listdir(p)
            )
        except Exception:
            has_jpg = False
        if has_jpg:
            test_dir = p
            break
if test_dir is None:
    test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images/"

sample_sub_path_candidates = [
    "/kaggle/input/cassava-leaf-disease-classification/sample_submission.csv",
    "/kaggle/input/sample_submission.csv",
    "/kaggle/data/cassava-leaf-disease-classification/sample_submission.csv",
    "/kaggle/data/sample_submission.csv",
]
sample_sub_path = None
for p in sample_sub_path_candidates:
    if os.path.exists(p):
        sample_sub_path = p
        break
if sample_sub_path is None:
    sample_sub_path = (
        "/kaggle/input/cassava-leaf-disease-classification/sample_submission.csv"
    )

train_csv_path_candidates = [
    "/kaggle/input/cassava-leaf-disease-classification/train.csv",
    "/kaggle/input/train.csv",
    "/kaggle/data/cassava-leaf-disease-classification/train.csv",
    "/kaggle/data/train.csv",
]
train_csv_path = None
for p in train_csv_path_candidates:
    if os.path.exists(p):
        train_csv_path = p
        break

# Score-relevant fix:
# Ensure the declared img_size matches the actual crop size produced by the chosen
# weights.transforms() pipeline. If these mismatch, ViT positional embedding interpolation
# / preprocess mismatch can significantly reduce accuracy.
from torchvision.models import vit_b_16, ViT_B_16_Weights

weights = ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1
_vit_preprocess = weights.transforms()


def _infer_transform_crop_size(transform_obj, default=384):
    # Try to infer the final (H,W) from torchvision v2 transforms without changing semantics.
    try:
        # Common pattern: Compose([... , CenterCrop(size), ...])
        if hasattr(transform_obj, "transforms"):
            for t in transform_obj.transforms:
                if isinstance(t, v2.CenterCrop):
                    sz = t.size
                    if isinstance(sz, int):
                        return int(sz)
                    if isinstance(sz, (tuple, list)) and len(sz) >= 2:
                        return int(sz[0])
        # Some pipelines may use RandomResizedCrop for training; here weights.transforms()
        # for ViT inference typically includes CenterCrop.
    except Exception:
        pass
    return int(default)


model_b_img_size = _infer_transform_crop_size(_vit_preprocess, default=384)
model_a_img_size = model_b_img_size

batch_size = 16
num_workers = 4
num_classes = 5
tta = False


def _safe_torch_load(path, map_location):
    try:
        if os.path.exists(path):
            return torch.load(path, map_location=map_location)
        return None
    except Exception:
        return None


def _find_checkpoint(filename, roots=("/kaggle/input", "/kaggle/data")):
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [
                d
                for d in dirnames
                if not d.startswith(".") and d not in ("__pycache__",)
            ]
            if filename in filenames:
                return os.path.join(dirpath, filename)
    return None


vit_v1_path = "/kaggle/input/vit-v1/vit_v1.pt"
vit_boosted_path = "/kaggle/input/vit-boosted/vit_boosted.pt"
linear_head_path = "/kaggle/input/linear-head/linear_cls.pt"

if not os.path.exists(vit_v1_path):
    found = _find_checkpoint("vit_v1.pt")
    if found is not None:
        vit_v1_path = found
if not os.path.exists(vit_boosted_path):
    found = _find_checkpoint("vit_boosted.pt")
    if found is not None:
        vit_boosted_path = found
if not os.path.exists(linear_head_path):
    found = _find_checkpoint("linear_cls.pt")
    if found is not None:
        linear_head_path = found

print("Checkpoint paths:")
print(" vit_v1_path     =", vit_v1_path, "| exists:", os.path.exists(vit_v1_path))
print(
    " vit_boosted_path=",
    vit_boosted_path,
    "| exists:",
    os.path.exists(vit_boosted_path),
)
print(
    " linear_head_path=",
    linear_head_path,
    "| exists:",
    os.path.exists(linear_head_path),
)
print("Using inferred preprocess crop size =", model_a_img_size)

model_a = _safe_torch_load(vit_v1_path, map_location=device)
model_b = _safe_torch_load(vit_boosted_path, map_location=device)
linear_head = _safe_torch_load(linear_head_path, map_location=device)

use_centroid_fallback = False
centroids = None  # shape [num_classes, D]
feat_dim = None

centroid_temp = 1.0

train_dir_candidates = [
    "/kaggle/input/cassava-leaf-disease-classification/train_images/",
    "/kaggle/input/cassava-leaf-disease-classification/train_images/train_images/",
    "/kaggle/input/train_images/",
    "/kaggle/input/train_images/train_images/",
    "/kaggle/data/cassava-leaf-disease-classification/train_images/",
    "/kaggle/data/cassava-leaf-disease-classification/train_images/train_images/",
    "/kaggle/data/train_images/",
    "/kaggle/data/train_images/train_images/",
]
train_dir = None
for p in train_dir_candidates:
    if os.path.isdir(p):
        try:
            has_jpg = any(
                os.path.isfile(os.path.join(p, f))
                and f.lower().endswith((".jpg", ".jpeg", ".png"))
                for f in os.listdir(p)
            )
        except Exception:
            has_jpg = False
        if has_jpg:
            train_dir = p
            break


def _strip_state_dict_prefixes(sd: dict):
    if not isinstance(sd, dict):
        return sd
    prefixes = ("module.", "model.", "net.", "backbone.")
    out = {}
    for k, v in sd.items():
        nk = k
        for p in prefixes:
            if nk.startswith(p):
                nk = nk[len(p) :]
        out[nk] = v
    return out


def _as_module(obj, arch_builder):
    if obj is None:
        return None
    if isinstance(obj, torch.nn.Module):
        return obj
    if isinstance(obj, dict):
        sd = obj.get("state_dict", obj)
        if isinstance(sd, dict):
            sd = _strip_state_dict_prefixes(sd)
            m = arch_builder()
            missing, unexpected = m.load_state_dict(sd, strict=False)
            if len(unexpected) > 0:
                print(
                    "Warning: unexpected keys when loading state_dict:", unexpected[:5]
                )
            if len(missing) > 0:
                print("Warning: missing keys when loading state_dict:", missing[:5])
            return m
    return None


def _build_vit5():
    m = vit_b_16(weights=None)
    if hasattr(m, "heads") and m.heads is not None:
        if (
            isinstance(m.heads, torch.nn.Sequential)
            and len(m.heads) > 0
            and isinstance(m.heads[-1], torch.nn.Linear)
        ):
            in_f = m.heads[-1].in_features
            m.heads[-1] = torch.nn.Linear(in_f, num_classes)
        else:
            hidden = getattr(m, "hidden_dim", 768)
            m.heads = torch.nn.Linear(hidden, num_classes)
    return m


model_a = _as_module(model_a, _build_vit5)
model_b = _as_module(model_b, _build_vit5)

if (model_a is None) or (model_b is None):
    backbone = vit_b_16(weights=weights).to(device)
    backbone.eval()
    use_centroid_fallback = True
else:
    backbone = None

if not use_centroid_fallback:
    model_a = model_a.to(device)
    model_b = model_b.to(device)
    model_a.eval()
    model_b.eval()

    if linear_head is not None and isinstance(linear_head, dict):
        pass
    elif linear_head is not None and hasattr(linear_head, "to"):
        linear_head = linear_head.to(device)
        linear_head.eval()

print("use_centroid_fallback =", use_centroid_fallback)




cuda
Checkpoint paths:
 vit_v1_path     = /kaggle/input/vit-v1/vit_v1.pt | exists: False
 vit_boosted_path= /kaggle/input/vit-boosted/vit_boosted.pt | exists: False
 linear_head_path= /kaggle/input/linear-head/linear_cls.pt | exists: False
Using inferred preprocess crop size = 384


Downloading: "https://download.pytorch.org/models/vit_b_16_swag-9ac1b537.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16_swag-9ac1b537.pth
100%|██████████| 331M/331M [00:03<00:00, 115MB/s]


use_centroid_fallback = True


In [2]:
class CassavaDataset(VisionDataset):
    """Custom dataset for the Cassava data.

    Uses the exact torchvision weights preprocessing pipeline via `weights.transforms()`.
    """

    def __init__(
        self,
        data_dir,
        model_a_size,
        model_b_size,
        transform=None,
        ttas=None,
        image_list=None,
    ):
        super().__init__(root=data_dir)

        self.transform = transform
        exts = (".jpg", ".jpeg", ".png")

        if image_list is not None:
            files = []
            for f in image_list:
                if not isinstance(f, str):
                    continue
                if f.startswith("."):
                    continue
                fp = os.path.join(data_dir, f)
                if os.path.isfile(fp) and f.lower().endswith(exts):
                    files.append(f)
            self.images = files
        else:
            files = []
            for f in os.listdir(data_dir):
                if f.startswith("."):
                    continue
                fp = os.path.join(data_dir, f)
                if os.path.isfile(fp) and f.lower().endswith(exts):
                    files.append(f)
            self.images = sorted(files)

        if len(self.images) == 0:
            raise RuntimeError(f"No image files found in test_dir={data_dir}")

        self.ttas = ttas

    def __getitem__(self, idx):
        filename = self.images[idx]
        img = Image.open(os.path.join(self.root, filename)).convert("RGB")

        if self.ttas is not None and self.transform is not None:
            model_a_img = [self.transform(t(img)) for t in self.ttas]
            model_b_img = [self.transform(t(img)) for t in self.ttas]
        elif self.transform:
            model_a_img = self.transform(img)
            model_b_img = self.transform(img)
        else:
            model_a_img = img
            model_b_img = img

        return model_a_img, model_b_img, filename

    def __len__(self):
        return len(self.images)




In [3]:
if tta:
    ttas = [
        v2.RandomRotation(180),
        v2.RandomVerticalFlip(1),
        v2.RandomPerspective(p=1),
    ]
else:
    ttas = None

sample_sub = pd.read_csv(sample_sub_path)
test_image_list = sample_sub["image_id"].tolist()

test_dataset = CassavaDataset(
    test_dir,
    model_a_img_size,
    model_b_img_size,
    transform=_vit_preprocess,
    ttas=ttas,
    image_list=test_image_list,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

normalizer = torch.nn.Softmax(dim=1)


def _vit_features(vit_model, x, apply_head: bool = True):
    vit_model.eval()
    with torch.no_grad():
        x = vit_model._process_input(x)
        n = x.shape[0]
        batch_class_token = vit_model.class_token.expand(n, -1, -1)
        x = torch.cat([batch_class_token, x], dim=1)
        x = vit_model.encoder(x)
        if hasattr(vit_model.encoder, "ln") and vit_model.encoder.ln is not None:
            x = vit_model.encoder.ln(x)
        x = x[:, 0]  # CLS token, [B, hidden_dim]

        if apply_head and hasattr(vit_model, "heads") and vit_model.heads is not None:
            x = vit_model.heads(x)
    return x


def _infer_linear_in_dim_from_sd(sd: dict):
    if not isinstance(sd, dict):
        return None
    sd = _strip_state_dict_prefixes(sd)
    for k, v in sd.items():
        if not torch.is_tensor(v):
            continue
        if k.endswith("weight") and v.ndim == 2 and v.shape[0] == num_classes:
            return int(v.shape[1])
    return None


def _ensure_linear_head(linear_head_obj, in_dim: int):
    if linear_head_obj is None:
        return None
    if isinstance(linear_head_obj, torch.nn.Module):
        return linear_head_obj
    if isinstance(linear_head_obj, dict):
        sd = linear_head_obj.get("state_dict", linear_head_obj)
        if isinstance(sd, dict):
            sd = _strip_state_dict_prefixes(sd)
            m = torch.nn.Linear(in_dim, num_classes)
            m.load_state_dict(sd, strict=False)
            return m
    return None


if "use_centroid_fallback" in globals() and use_centroid_fallback:
    if train_csv_path is None or train_dir is None:
        raise RuntimeError(
            "Centroid fallback enabled but train.csv or train_images directory not found."
        )

    train_df_full = pd.read_csv(train_csv_path)

    g = torch.Generator().manual_seed(3407)
    labels_t = torch.tensor(train_df_full["label"].values, dtype=torch.long)
    calib_mask = torch.zeros(len(train_df_full), dtype=torch.bool)
    for c in range(num_classes):
        idx = (labels_t == c).nonzero(as_tuple=False).view(-1)
        if len(idx) == 0:
            continue
        perm = idx[torch.randperm(len(idx), generator=g)]
        take = max(1, int(0.1 * len(perm)))  # 10% per class for calibration
        calib_mask[perm[:take]] = True

    calib_df = train_df_full.loc[calib_mask.numpy()].reset_index(drop=True)
    build_df = train_df_full.loc[(~calib_mask).numpy()].reset_index(drop=True)

    class_sums = None
    class_counts = torch.zeros(num_classes, dtype=torch.long, device=device)

    train_ds = CassavaDataset(
        train_dir,
        model_a_img_size,
        model_b_img_size,
        transform=_vit_preprocess,
        ttas=None,
        image_list=build_df["image_id"].tolist(),
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    labels_map = dict(zip(build_df["image_id"].tolist(), build_df["label"].tolist()))

    backbone.eval()
    for model_a_inputs, _, filenames in train_loader:
        model_a_inputs = model_a_inputs.to(device, non_blocking=True)

        feats = _vit_features(backbone, model_a_inputs, apply_head=False)  # [B,D]
        if class_sums is None:
            feat_dim = feats.shape[1]
            class_sums = torch.zeros(num_classes, feat_dim, device=device)

        for i, fn in enumerate(filenames):
            y = int(labels_map[str(fn)])
            class_sums[y] += feats[i]
            class_counts[y] += 1

    class_counts_safe = class_counts.clamp(min=1).unsqueeze(1)
    centroids = class_sums / class_counts_safe
    centroids = torch.nn.functional.normalize(centroids, dim=1)
    print("Built centroids with counts:", class_counts.detach().cpu().tolist())

    calib_ds = CassavaDataset(
        train_dir,
        model_a_img_size,
        model_b_img_size,
        transform=_vit_preprocess,
        ttas=None,
        image_list=calib_df["image_id"].tolist(),
    )
    calib_loader = DataLoader(
        calib_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    calib_labels_map = dict(
        zip(calib_df["image_id"].tolist(), calib_df["label"].tolist())
    )

    sims_list = []
    y_list = []
    backbone.eval()
    with torch.no_grad():
        for model_a_inputs, _, filenames in calib_loader:
            model_a_inputs = model_a_inputs.to(device, non_blocking=True)
            feats = _vit_features(backbone, model_a_inputs, apply_head=False)
            feats = torch.nn.functional.normalize(feats, dim=1)
            sims = feats @ centroids.T  # [B,C]
            sims_list.append(sims.detach().float().cpu())
            for fn in filenames:
                y_list.append(int(calib_labels_map[str(fn)]))

    sims_all = torch.cat(sims_list, dim=0)  # CPU float32
    y_all = torch.tensor(y_list, dtype=torch.long)

    logT = torch.tensor(0.0, requires_grad=True)  # T starts at 1.0
    opt = torch.optim.LBFGS([logT], lr=0.25, max_iter=25, line_search_fn="strong_wolfe")

    def _closure():
        opt.zero_grad(set_to_none=True)
        T = torch.exp(logT).clamp(0.25, 8.0)
        logits = sims_all / T
        loss = torch.nn.functional.cross_entropy(logits, y_all)
        loss.backward()
        return loss

    opt.step(_closure)
    centroid_temp = float(torch.exp(logT).clamp(0.25, 8.0).detach().cpu().item())
    print("Fitted centroid temperature:", centroid_temp)

linear_head_module = None
linear_head_in_dim = None
if not ("use_centroid_fallback" in globals() and use_centroid_fallback):
    try:
        inferred_in_dim = None
        if isinstance(linear_head, dict):
            sd0 = linear_head.get("state_dict", linear_head)
            inferred_in_dim = (
                _infer_linear_in_dim_from_sd(sd0) if isinstance(sd0, dict) else None
            )
        if inferred_in_dim is None:
            inferred_in_dim = 768

        linear_head_module = _ensure_linear_head(linear_head, in_dim=inferred_in_dim)
        if linear_head_module is not None:
            linear_head_module = linear_head_module.to(device).eval()
            linear_head_in_dim = int(inferred_in_dim)
            print("Prepared linear_head_module with in_dim =", linear_head_in_dim)
        else:
            print("No usable linear_head_module; will prefer model forward logits.")
    except Exception as e:
        print("Failed to init linear_head_module:", repr(e))
        linear_head_module = None
        linear_head_in_dim = None

use_external_head_for_a = False
if not ("use_centroid_fallback" in globals() and use_centroid_fallback):
    if (
        "linear_head_module" in globals()
        and linear_head_module is not None
        and linear_head_in_dim is not None
    ):
        try:
            dummy = torch.zeros(1, 3, model_a_img_size, model_a_img_size, device=device)
            feat_probe = _vit_features(model_a, dummy, apply_head=False)
            use_external_head_for_a = int(feat_probe.shape[1]) == int(
                linear_head_in_dim
            )
        except Exception:
            use_external_head_for_a = False
    print("use_external_head_for_a =", use_external_head_for_a)



Built centroids with counts: [846, 1711, 1882, 10371, 2041]
Fitted centroid temperature: 0.25


In [4]:
all_names = []
all_preds = []

if "use_centroid_fallback" in globals() and use_centroid_fallback:
    backbone.eval()
else:
    model_a.eval()
    model_b.eval()
    if "linear_head_module" in globals() and linear_head_module is not None:
        linear_head_module.eval()
    elif linear_head is not None and hasattr(linear_head, "eval"):
        linear_head.eval()

with torch.no_grad():
    for _, (model_a_inputs, model_b_inputs, filenames) in enumerate(test_loader):
        if tta:
            batch_n = len(filenames)
            model_a_inputs = torch.cat(model_a_inputs, dim=0).to(
                device, non_blocking=True
            )
            model_b_inputs = torch.cat(model_b_inputs, dim=0).to(
                device, non_blocking=True
            )
            filenames = list(filenames)

            if "use_centroid_fallback" in globals() and use_centroid_fallback:
                feats = _vit_features(backbone, model_a_inputs, apply_head=False)
                feats = torch.nn.functional.normalize(feats, dim=1)
                sims = (feats @ centroids.T) / float(centroid_temp)  # calibrated
                sims_batch = torch.stack(torch.split(sims, batch_n), dim=0)
                sims_mean = torch.mean(sims_batch, dim=0)
                pred_labels = torch.argmax(sims_mean, 1).tolist()
            else:
                if use_external_head_for_a:
                    feats_a = _vit_features(model_a, model_a_inputs, apply_head=False)
                    logits_a = linear_head_module(feats_a)
                    logits_b = model_b(model_b_inputs)
                else:
                    logits_a = model_a(model_a_inputs)
                    logits_b = model_b(model_b_inputs)

                logits_a_batch = torch.stack(torch.split(logits_a, batch_n), dim=0)
                logits_a_mean = torch.mean(logits_a_batch, dim=0)

                logits_b_batch = torch.stack(torch.split(logits_b, batch_n), dim=0)
                logits_b_mean = torch.mean(logits_b_batch, dim=0)

                outputs = 0.9 * logits_a_mean + 0.1 * logits_b_mean
                preds = normalizer(outputs)
                pred_labels = torch.argmax(preds, 1).tolist()
        else:
            model_a_inputs = model_a_inputs.to(device, non_blocking=True)
            model_b_inputs = model_b_inputs.to(device, non_blocking=True)
            filenames = list(filenames)

            if "use_centroid_fallback" in globals() and use_centroid_fallback:
                feats = _vit_features(backbone, model_a_inputs, apply_head=False)
                feats = torch.nn.functional.normalize(feats, dim=1)
                sims = (feats @ centroids.T) / float(centroid_temp)  # calibrated
                pred_labels = torch.argmax(sims, 1).tolist()
            else:
                if use_external_head_for_a:
                    feats_a = _vit_features(model_a, model_a_inputs, apply_head=False)
                    model_a_outputs = linear_head_module(feats_a)
                    model_b_outputs = model_b(model_b_inputs)
                else:
                    model_a_outputs = model_a(model_a_inputs)
                    model_b_outputs = model_b(model_b_inputs)

                outputs = 0.9 * model_a_outputs + 0.1 * model_b_outputs
                preds = normalizer(outputs)
                pred_labels = torch.argmax(preds, 1).tolist()

        all_names.extend(filenames)
        all_preds.extend(pred_labels)

pred_map = dict(zip(all_names, all_preds))



In [5]:
ordered_preds = sample_sub["image_id"].map(pred_map)
ordered_preds = ordered_preds.fillna(0).astype(int)

my_submission = pd.DataFrame(
    {"image_id": sample_sub["image_id"], "label": ordered_preds}
)

assert len(my_submission) == len(sample_sub), (len(my_submission), len(sample_sub))
assert my_submission["image_id"].isna().sum() == 0
assert my_submission["label"].between(0, 4).all()

my_submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", my_submission.shape)
print(my_submission.head())

Wrote submission.csv with shape: (2676, 2)
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3
